# P24 — Representaciones profundas de palabras dependientes del contexto

## 1. Título y paper

**Paper:** *Deep Contextualized Word Representations*  
**Autoría:** Matthew E. Peters, Mark Neumann, Mohit Iyyer, Matt Gardner, Christopher Clark, Kenton Lee, Luke Zettlemoyer  
**Año y venue:** 2018 · NAACL 2018 · ACL Anthology N18-1202  
**Nivel:** L3 · **Motor:** `elmo`  
**Ficha completa:** [`P24_elmo`](../../papers/foundational/P24_elmo/README.md)

**Hito:** Un vector por APARICIÓN y no por palabra: la polisemia deja de colapsar en un único punto del espacio.

- [ACL Anthology (NAACL 2018)](https://aclanthology.org/N18-1202/)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Un embedding estático da el mismo vector a «banco del parque» y «banco central»: el sentido se pierde antes de que el modelo empiece a trabajar.
2. Ejecutar una implementación mínima de la propuesta: Usar los estados internos de un modelo de lenguaje bidireccional profundo y combinar sus capas con pesos aprendidos por tarea.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P03
- P05
- P23


## 4. Intuición

Un diccionario da una entrada por palabra. Un lector da un significado por aparición. ELMo deja de ser diccionario: el vector de «banco» se calcula leyendo la frase entera.


## 5. Concepto mínimo

```text
Estático (P05, P23):   v(banco) = siempre el mismo vector

ELMo:  ELMo_k = γ · Σ_j s_j · h_{k,j}

    h_{k,j} = estado de la capa j del LM bidireccional en la posición k
    s_j     = pesos por capa, APRENDIDOS para cada tarea
```

Las capas bajas capturan sintaxis y las altas semántica, así que cada tarea aprende **cuánto pesa cada capa** en vez de recibir una mezcla fija.


## 6. Código explicado

El motor calcula el vector de «banco» en tres frases con sentidos distintos, de forma estática y contextual.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('elmo', seed=7)['result']
for c in r['contextos']:
    print(' ·', c)
print('\nestático   :', r['similitud_estatica'])
print('contextual :', r['similitud_contextual'])

## 7. Predicción antes de ejecutar

1. ¿Cuánto valdrá el coseno entre las tres apariciones con embedding estático?
2. ¿Qué dos sentidos deberían quedar más cerca entre sí: parque/río o parque/central?
3. ¿Por qué combinar varias capas en vez de usar solo la última?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
r = run_paper_lab('elmo', seed=7)['result']
e, c = r['similitud_estatica'], r['similitud_contextual']
for par in e:
    print(f'{par:<20} estático {e[par]:+.3f} → contextual {c[par]:+.3f} '
          f'(separación {e[par] - c[par]:+.3f})')

## 9. Salida interpretable

Con embedding estático los tres cosenos valen 1,0: son literalmente el mismo vector. Con representación contextual bajan, y bajan **de forma desigual**: los sentidos parecidos quedan más cerca que los distantes. Eso es lo que un clasificador aguas abajo puede aprovechar.


## 10. Comentario pedagógico

ELMo se usaba como **características congeladas**: se calculaban los vectores y se alimentaban a un modelo específico de tarea. BERT, meses después, ajustaría el modelo entero. La diferencia entre «extraer características» y «ajustar todo» define dos épocas del PLN.


## 11. Error o anti-patrón deliberado

Anti-patrón: usar solo la última capa porque «es la más profunda».


In [ ]:
capas = {'capa 0 (tokens)': 'morfología y ortografía',
         'capa 1 (baja)': 'sintaxis: categoría gramatical, dependencias',
         'capa 2 (alta)': 'semántica: sentido en contexto'}
for k, v in capas.items():
    print(f'{k:<18} → {v}')
print('\nUna tarea de etiquetado gramatical quiere la capa baja, no la alta.')

## 12. Corrección

La corrección es dejar que la tarea decida los pesos por capa:


In [ ]:
import math
for tarea, s in (('etiquetado gramatical', [0.2, 0.6, 0.2]),
                 ('respuesta a preguntas', [0.1, 0.3, 0.6])):
    print(f'{tarea:<24} pesos por capa {s} (suman {sum(s):.1f})')

## 13. Desafío guiado

Añade una cuarta frase con «banco» en el sentido de asiento y comprueba con cuál de las tres se agrupa.


In [ ]:
r = run_paper_lab('elmo', seed=7)['result']
print('sentidos distinguidos:', r['sentidos_distinguidos'])
print('el par más separado es el de sentidos más distintos:')
for par, v in sorted(r['similitud_contextual'].items(), key=lambda kv: kv[1]):
    print(f'  {par:<20} {v:+.3f}')

## 14. Desafío autónomo

Con un modelo contextual abierto y ejecutable localmente, toma 30 frases con una palabra polisémica y agrupa sus vectores. Comprueba si los grupos se corresponden con los sentidos del diccionario y documenta los casos donde no.


## 15. Evidencia de aprendizaje

Guarda las similitudes estática y contextual, la explicación de por qué se combinan capas y tu cuarta frase con el resultado de agrupamiento.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P24_elmo/README.md) · evaluación formal: [`assessments/papers/P24_elmo.md`](../../assessments/papers/P24_elmo.md)


## 16. Cierre

Las representaciones ya dependen del contexto. Falta unificar **las tareas**: cada una seguía necesitando su propia cabeza y su propio formato.


## 17. Conexión con el siguiente hito

- P09
- P25

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
